# OptiChat prompts on open models (Qwen 2.5 + Llama 3.1) — Colab replication of Appendix C

Reconstruction of the agent prompts described in Appendix C of Chen et al., *OptiChat: Bridging Optimization Models and Practitioners with LLMs*
(INFORMS Journal on Data Science, 2026, doi:10.1287/ijds.2025.0074), run on **Qwen2.5-7B-Instruct** and **Llama-3.1-8B-Instruct** only.

**What this notebook runs (one section per Appendix C item)**

| § | Appendix C item | What we test |
|---|---|---|
| 1 | component interpretation | JSON name→description for every model component (coverage) |
| 2 | model illustration | plain-English intro from the JSON model representation |
| 3 | IIS inference | explain an infeasible model from its IIS |
| 4 | coordinator | routing JSON, incl. the `DONE` signal |
| 5 | explainer | markdown answer from tool output; **numbers must be grounded in tool output** |
| 6 | syntax reminder | choose 1 of 5 functions + component |
| 7 | operator | emit tool-call arguments, then **really execute** the solver tool |
| 8 | code reminder | program scaffold with the `# YOUR CODE GOES HERE` marker |
| 9 | programmer (+ its 2 worked examples) | write counterfactual constraints; objective must match ground truth |
| 10 | evaluator | accept/reject JSON on good, buggy and (acceptably) infeasible code |
| 11 | full loop | coordinator → engineer → explainer end to end |

**Honest caveats.** (1) Prompts are written in my own words from the roles the paper describes; the paper's exact text is not reproduced here (publisher copyright) —
paste your own copy into the optional override cell if you need strict replication. (2) Toy models, not the paper's 24-model benchmark. (3) Small n: treat the
scoreboard as a smoke test, not a result. (4) Counterfactual code runs with restricted builtins — **not a security sandbox**.

**To run:** Runtime → Change runtime type → **T4 GPU**. Qwen needs nothing. **Llama 3.1 is gated**: accept Meta's license at
huggingface.co/meta-llama/Llama-3.1-8B-Instruct, create a token, and add it in Colab *Secrets* (key `HF_TOKEN`, notebook access ON). Never paste a token into a cell you will share.
Llama 3.1 is under the Llama 3.1 Community License ("Built with Llama"); Qwen2.5-7B-Instruct is Apache-2.0.

In [ ]:
!pip -q install pyomo highspy transformers accelerate bitsandbytes pandas

In [ ]:
# ---- Only these two model families are supported by this notebook ----
import os, gc, json, re, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_IDS = {"qwen": "Qwen/Qwen2.5-7B-Instruct", "llama": "meta-llama/Llama-3.1-8B-Instruct"}
USE = ["qwen", "llama"]          # e.g. ["qwen"] to skip the gated Llama download
FAST = False                      # True skips the §11 end-to-end loop (saves ~10 min)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
LLMS = {}
for k in USE:
    try:
        tok = AutoTokenizer.from_pretrained(MODEL_IDS[k], token=HF_TOKEN)
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        mdl = AutoModelForCausalLM.from_pretrained(MODEL_IDS[k], quantization_config=bnb, device_map="auto", token=HF_TOKEN)
        mdl.eval(); LLMS[k] = (tok, mdl); print("loaded", k, MODEL_IDS[k])
    except Exception as e:
        print(f"!! could not load {k}: {type(e).__name__}: {str(e)[:200]}\n   (Llama: accept the license + set HF_TOKEN secret.) Continuing without it.")
LLMS_KEYS = list(LLMS)
assert LLMS_KEYS, "No model loaded"

In [ ]:
def generate(key, role, system, user, history=None, max_new_tokens=600):
    """Greedy decoding (deterministic). `role` is only a label (used for logging / tests)."""
    tok, mdl = LLMS[key]
    msgs = [{"role": "system", "content": system}] + (history or []) + [{"role": "user", "content": user}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re, json, time, math, io, contextlib
import pandas as pd

def extract_json(text):
    """First balanced {...} in an LLM reply (models wrap JSON in prose / markdown fences)."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    start = text.find("{")
    while start != -1:
        depth = 0
        for i in range(start, len(text)):
            depth += (text[i] == "{") - (text[i] == "}")
            if depth == 0:
                try: return json.loads(text[start:i + 1])
                except json.JSONDecodeError: break
        start = text.find("{", start + 1)
    raise ValueError("no JSON found: " + text[:150])

def extract_code(text):
    m = re.search(r"```(?:python)?\n(.*?)```", text, flags=re.S)
    return (m.group(1) if m else text).strip()

RESULTS = []   # scoreboard rows
def record(section, model, case, passed, note=""):
    RESULTS.append(dict(section=section, model=model, case=case, passed=bool(passed), note=str(note)[:120]))

def run_all(role, system, user, history=None, max_new_tokens=600):
    out = {}
    for k in LLMS_KEYS:
        t0 = time.time(); out[k] = generate(k, role, system, user, history, max_new_tokens); out[k + "_sec"] = round(time.time() - t0, 1)
    return out

def show(title, outs, maxc=900):
    print("=" * 100); print(title)
    for k in LLMS_KEYS:
        print(f"--- {k} ({outs[k+'_sec']}s) ---"); print(outs[k][:maxc] + (" …" if len(outs[k]) > maxc else ""))

## 0. Solver tools and toy models
The "predefined functions" are plain Pyomo + HiGHS (no LLM): diagnosing (IIS + elastic relaxation), retrieval, sensitivity (LP duals), what-if (re-solve), and a
counterfactual runner where the LLM writes **constraints only** and our code does the solving. These were tested against an independent scipy LP, finite differences, and a known IIS.

In [ ]:
# Solver tools (Pyomo + HiGHS/Gurobi). No LLM code here. Tested: scipy LP cross-check, finite-difference sensitivity, known IIS.
"""The 'predefined functions' (paper Sec. 3.1). Pure Pyomo + solver; NO LLM in this file.
Every function returns a JSON-serialisable dict so the Explainer can narrate it."""
import os
import pyomo.environ as pyo
from pyomo.core.expr.visitor import replace_expressions, identify_mutable_parameters
from pyomo.opt import TerminationCondition as TC


# ---------------------------------------------------------------- solving
def get_solver():
    names = [os.environ["OPTICHAT_SOLVER"]] if os.environ.get("OPTICHAT_SOLVER") else ["gurobi_direct", "appsi_highs"]
    for n in names:
        s = pyo.SolverFactory(n)
        if s.available(exception_flag=False):
            return n, s
    raise RuntimeError("No solver found (install gurobipy or highspy)")

def solve(model, duals=False):
    """Solve a (clone of a) model. Returns (status_str, objective_or_None)."""
    if duals and not hasattr(model, "dual"):
        model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    name, s = get_solver()
    if name == "gurobi_direct":
        s.options["DualReductions"] = 0      # report 'infeasible' vs 'unbounded' unambiguously
    try:
        r = s.solve(model, load_solutions=False) if name == "gurobi_direct" else s.solve(model)
    except Exception as e:  # some solvers raise when infeasible and no solution can be loaded
        msg = str(e).lower()
        if "infeasible" in msg: return "infeasible", None
        if "feasible solution was not found" in msg: return "infeasible_or_no_solution", None   # HiGHS (appsi) wording
        return "error: " + str(e)[:120], None
    tc = r.solver.termination_condition if hasattr(r, "solver") else None
    if name == "gurobi_direct":
        if tc == TC.optimal:
            model.solutions.load_from(r)
    if tc in (TC.optimal,):
        obj = next(model.component_data_objects(pyo.Objective, active=True))
        return "optimal", pyo.value(obj)
    return str(tc), None

def is_lp(model):
    return not any(v.is_integer() or v.is_binary() for v in model.component_data_objects(pyo.Var))

def _get(model, name):
    comp = getattr(model, name, None)
    if comp is None:
        raise KeyError(f"unknown component '{name}'")
    return comp

def _idx(comp, index):
    """JSON list -> Pyomo index (scalar, or tuple)."""
    if index is None or index == [] or index == ():
        return None
    if isinstance(index, (list, tuple)):
        index = tuple(index) if len(index) > 1 else index[0]
    return index


# ---------------------------------------------------------------- IIS (deletion filter, Chinneck & Dravnieks 1991)
def _feasible(model):
    c = model.clone()
    st, _ = solve(c)
    return st == "optimal"

def compute_iis(model):
    """Solver-agnostic deletion filter over constraints (variable bounds are not examined).
    With a full Gurobi licence you can swap this for Model.computeIIS()."""
    m = model.clone()
    for o in m.component_data_objects(pyo.Objective, active=True):
        o.deactivate()                       # pure feasibility problem; also avoids 'unbounded' when constraints are removed
    m._feas_obj = pyo.Objective(expr=0)
    cons = list(m.component_data_objects(pyo.Constraint, active=True))
    if _feasible(m):
        return []
    keep = list(cons)
    for c in cons:
        c.deactivate()
        if _feasible(m):      # removing c made it feasible -> c belongs to the IIS
            c.activate()
        else:                 # still infeasible without c -> c is not needed
            keep.remove(c)
    return [c.name for c in keep]


# ---------------------------------------------------------------- 1) Diagnosing: elastic RHS/param relaxation  (Eq. 2)
def feasibility_restoration(model, component, index=None, direction="both"):
    """Relax a mutable Param (default: both ways) by slack variables and minimise total perturbation.
    direction: 'up' (paper's  b + d+), 'down', or 'both'. Param on the LHS of a constraint is refused (MIQCP)."""
    p = _get(model, component)
    if not isinstance(p, pyo.Param) or not p.mutable:
        return {"error": f"'{component}' is not a mutable Param"}
    m = model.clone()
    p = getattr(m, component)
    keys = list(p.keys()) if index in (None, [], "all") else [_idx(p, index)]
    # refuse LHS parameters: bilinear slack*variable terms => nonconvex MIQCP
    for c in m.component_data_objects(pyo.Constraint, active=True):
        if any(id(p[k]) in {id(q) for q in identify_mutable_parameters(c.body)} for k in keys if k in p):
            return {"error": f"'{component}' multiplies decision variables in '{c.name}' (LHS). Adding slack would create a "
                             "nonconvex MIQCP and LHS data are usually physically immutable; not performed."}
    m.dp = pyo.Var(keys, domain=pyo.NonNegativeReals)
    m.dn = pyo.Var(keys, domain=pyo.NonNegativeReals)
    if direction == "up":
        for k in keys: m.dn[k].fix(0)
    if direction == "down":
        for k in keys: m.dp[k].fix(0)
    smap = {id(p[k]): p[k] + m.dp[k] - m.dn[k] for k in keys}
    for c in list(m.component_data_objects(pyo.Constraint, active=True)):
        if any(id(q) in smap for q in identify_mutable_parameters(c.expr)):
            c.set_value(replace_expressions(c.expr, smap))
    for o in m.component_data_objects(pyo.Objective, active=True):
        o.deactivate()
    m.elastic_obj = pyo.Objective(expr=sum(m.dp[k] + m.dn[k] for k in keys))
    st, val = solve(m)
    if st != "optimal":
        return {"status": st, "message": "Even unlimited change of this parameter cannot restore feasibility."}
    delta = {k: pyo.value(m.dp[k]) - pyo.value(m.dn[k]) for k in keys}
    changes = {str(k): round(d, 6) for k, d in delta.items() if abs(d) > 1e-7}
    return {"status": "optimal", "component": component, "total_adjustment": round(val, 6),
            "required_changes(+increase/-decrease)": changes,
            "original_values": {str(k): pyo.value(p[k]) for k in keys if str(k) in changes}}


# ---------------------------------------------------------------- 2) Retrieval
def components_retrieval(model, component, index=None):
    comp = _get(model, component)
    st, obj = solve(model)
    out = {"component": component, "solution_status": st}
    def val(d):
        if isinstance(d, pyo.Constraint._ComponentDataClass) or hasattr(d, "body"):
            return {"lower": pyo.value(d.lower) if d.lower is not None else None,
                    "upper": pyo.value(d.upper) if d.upper is not None else None}
        if isinstance(d, pyo.Set) or hasattr(d, "ordered_data"):
            return list(d)
        return pyo.value(d, exception=False)
    if isinstance(comp, pyo.Set):
        out["values"] = [list(x) if isinstance(x, tuple) else x for x in comp]; return out
    if index in (None, [], "all") or (isinstance(index, list) and any(i in (":", "all") for i in index)):
        out["values"] = {str(k): val(v) for k, v in comp.items()} if comp.is_indexed() else val(comp)
    else:
        k = _idx(comp, index)
        if isinstance(index, list) and any(i in (":", "all") for i in index):
            pass
        out["values"] = {str(k): val(comp[k])}
    return out


# ---------------------------------------------------------------- 3) Sensitivity via LP duality (Eq. 3)
def sensitivity_analysis(model, component, index=None):
    if not is_lp(model):
        return {"error": "Sensitivity (dual) analysis is only valid for LPs; this model has integer variables. "
                         "Please state a concrete change and I will answer it as a what-if."}
    p = _get(model, component)
    if not isinstance(p, pyo.Param):
        return {"error": f"'{component}' is not a Param"}
    m = model.clone(); p = getattr(m, component)
    st, obj = solve(m, duals=True)
    if st != "optimal":
        return {"status": st, "message": "Model not optimal; use diagnosing first."}
    keys = list(p.keys()) if index in (None, [], "all") else [_idx(p, index)]
    res = {}
    obj_pids = {id(q) for o in m.component_data_objects(pyo.Objective, active=True) for q in identify_mutable_parameters(o.expr)}
    if any(id(p[k]) in obj_pids for k in keys):
        return {"error": f"'{component}' is an objective coefficient, not a right-hand side; duals do not give its sensitivity. "
                         "Please state a concrete change and I will answer it as a what-if."}
    for k in keys:
        pid = id(p[k]); tot = 0.0; used = False
        for c in m.component_data_objects(pyo.Constraint, active=True):
            in_body = pid in {id(q) for q in identify_mutable_parameters(c.body)}
            in_rhs = pid in {id(q) for q in identify_mutable_parameters(c.expr)} and not in_body
            if in_body:
                return {"error": f"'{component}' is a left-hand-side coefficient in '{c.name}'; its effect cannot be read from duals."}
            if in_rhs:
                used = True
                # d obj / d rhs = dual (Pyomo import suffix sign convention verified against finite differences in test_tools.py)
                tot += m.dual[c]
        res[str(k)] = round(tot, 6) if used else 0.0
    return {"status": "optimal", "objective": obj, "component": component,
            "d_objective_per_unit_increase": res,
            "note": "Local (valid while the optimal basis is unchanged)."}


# ---------------------------------------------------------------- 4) What-if (Eq. 4)
def evaluate_modification(model, component, index, operation, delta):
    """Apply Param change (operation in '+', '-', '*', 'set'), re-solve from scratch."""
    base_status, base_obj = solve(model.clone())
    m = model.clone(); p = getattr(m, component)
    if not isinstance(p, pyo.Param):
        return {"error": f"'{component}' is not a Param"}
    keys = list(p.keys()) if index in (None, [], "all") else [_idx(p, index)]
    for k in (keys if p.is_indexed() else [None]):
        d = p[k] if k is not None else p
        old = pyo.value(d)
        new = {"+": old + delta, "-": old - delta, "*": old * delta, "set": delta}[operation]
        d.value = new
    st, obj = solve(m)
    return {"component": component, "index": index, "operation": operation, "delta": delta,
            "status": st, "objective_before": base_obj, "objective_after": obj,
            "change": None if obj is None or base_obj is None else obj - base_obj}


# ---------------------------------------------------------------- 5) Why-not / unclassified: LLM-written code, OUR solve block
def run_counterfactual(model, snippet, solve_first=False):
    """Execute LLM-written lines (add constraints and/or print) on a clone, then re-solve with OUR solver code.
    The LLM never writes solver/solve code (paper's 'standard code' trick). solve_first=True loads the base solution
    (and duals if LP) before the snippet runs, so it can print values -- used by the no-predefined-functions ablation."""
    import io, contextlib
    base_status, base_obj = solve(model.clone())
    m = model.clone()
    if solve_first:
        solve(m, duals=is_lp(m))
    buf = io.StringIO()
    names = ("sum", "range", "len", "min", "max", "abs", "any", "all", "enumerate", "float", "int", "list", "set", "zip", "round", "str", "dict", "sorted", "print")
    bi = __builtins__ if isinstance(__builtins__, dict) else vars(__builtins__)
    ns = {"__builtins__": {k: bi[k] for k in names}, "model": m, "pyo": pyo, "value": pyo.value,
          "ConstraintList": pyo.ConstraintList, "Constraint": pyo.Constraint}
    # NOTE: restricted builtins deter accidents (import/open); this is NOT a security sandbox - run untrusted LLM output in a container.
    with contextlib.redirect_stdout(buf):
        exec(compile(snippet, "<llm_code>", "exec"), ns)
    st, obj = solve(m)
    return {"status": st, "objective_before": base_obj, "objective_after": obj, "printed": buf.getvalue()[-2000:]}


Two toy models stand in for the paper's examples. Their data live in mutable Params with a short comment on each line (our fallback "descriptions").

* **coffee** (MILP): suppliers → roasteries with contract fees and binary links `z` (the Appendix C worked example 1 is about forcing a single supplier for a roastery).
* **plant** (LP): production + transportation cost (worked example 2 compares the two cost terms).

In [ ]:
MODEL_SRC = r"""
import pyomo.environ as pyo

def build_coffee(infeasible=False):
    m = pyo.ConcreteModel()
    m.S = pyo.Set(initialize=["S1", "S2", "S3"])             # coffee bean suppliers
    m.R = pyo.Set(initialize=["roastery1", "roastery2"])     # roasteries
    m.capacity_in_supplier = pyo.Param(m.S, initialize={"S1": 450, "S2": 350, "S3": 600}, mutable=True)   # kg of beans each supplier can ship in total
    m.need = pyo.Param(m.R, initialize={"roastery1": 400, "roastery2": 1500 if infeasible else 550}, mutable=True)   # kg of beans each roastery needs
    m.unit_cost = pyo.Param(m.S, m.R, initialize={("S1","roastery1"): 2.0, ("S1","roastery2"): 2.2, ("S2","roastery1"): 2.4,
        ("S2","roastery2"): 2.1, ("S3","roastery1"): 3.0, ("S3","roastery2"): 3.2}, mutable=True)   # purchase plus shipping cost per kg
    m.link_fee = pyo.Param(initialize=100.0, mutable=True)   # fixed fee for each active supplier-roastery contract
    m.x = pyo.Var(m.S, m.R, domain=pyo.NonNegativeReals)     # kg shipped from supplier to roastery
    m.z = pyo.Var(m.S, m.R, domain=pyo.Binary)               # 1 if supplier serves roastery
    m.meet_need = pyo.Constraint(m.R, rule=lambda m, r: sum(m.x[s, r] for s in m.S) == m.need[r])
    m.supplier_cap = pyo.Constraint(m.S, rule=lambda m, s: sum(m.x[s, r] for r in m.R) <= m.capacity_in_supplier[s])
    m.link = pyo.Constraint(m.S, m.R, rule=lambda m, s, r: m.x[s, r] <= m.capacity_in_supplier[s] * m.z[s, r])
    m.total_cost = pyo.Objective(sense=pyo.minimize,
        expr=sum(m.unit_cost[s, r] * m.x[s, r] for s in m.S for r in m.R) + m.link_fee * sum(m.z[s, r] for s in m.S for r in m.R))
    return m

def build_plant(infeasible=False):
    m = pyo.ConcreteModel()
    m.P = pyo.Set(initialize=["plantA", "plantB"])            # plants
    m.D = pyo.Set(initialize=["east", "west", "north"])       # demand regions
    m.prod_cost = pyo.Param(m.P, initialize={"plantA": 4.0, "plantB": 6.0}, mutable=True)   # production cost per unit
    m.ship_cost = pyo.Param(m.P, m.D, initialize={("plantA","east"): 1, ("plantA","west"): 6, ("plantA","north"): 9,
        ("plantB","east"): 1, ("plantB","west"): 3, ("plantB","north"): 9}, mutable=True)   # transportation cost per unit
    m.plant_cap = pyo.Param(m.P, initialize={"plantA": 900, "plantB": 700}, mutable=True)   # units each plant can make
    m.demand = pyo.Param(m.D, initialize={"east": 300, "west": 400, "north": 2000 if infeasible else 500}, mutable=True)   # units each region needs
    m.ship = pyo.Var(m.P, m.D, domain=pyo.NonNegativeReals)   # units shipped from plant to region
    m.meet = pyo.Constraint(m.D, rule=lambda m, d: sum(m.ship[p, d] for p in m.P) == m.demand[d])
    m.cap = pyo.Constraint(m.P, rule=lambda m, p: sum(m.ship[p, d] for d in m.D) <= m.plant_cap[p])
    m.production = pyo.Expression(expr=sum(m.prod_cost[p] * m.ship[p, d] for p in m.P for d in m.D))     # total production cost
    m.transportation = pyo.Expression(expr=sum(m.ship_cost[p, d] * m.ship[p, d] for p in m.P for d in m.D))   # total transportation cost
    m.total_cost = pyo.Objective(sense=pyo.minimize, expr=m.production + m.transportation)
    return m
"""
ns = {}; exec(MODEL_SRC, ns)
BUILD = {"coffee": lambda: ns["build_coffee"](), "coffee_infeasible": lambda: ns["build_coffee"](True),
         "plant": lambda: ns["build_plant"](), "plant_infeasible": lambda: ns["build_plant"](True)}
for k, f in BUILD.items():
    st, obj = solve(f().clone()); print(f"{k:18s} status={st:28s} objective={obj}")

In [ ]:
# ---- Illustrator helpers: component table + JSON representation (input to several Appendix C prompts) ----
from pyomo.core.expr.visitor import identify_mutable_parameters

def code_comments(src):
    out = {}
    for line in src.splitlines():
        mm = re.match(r"\s*(?:m|model)\.(\w+)\s*=.*#\s*(.+)$", line)
        if mm: out[mm.group(1)] = mm.group(2).strip()
    return out
DESC = code_comments(MODEL_SRC)

def json_representation(model, name):
    lhs = set()
    for c in model.component_data_objects(pyo.Constraint, active=True):
        lhs |= {q.parent_component().name for q in identify_mutable_parameters(c.body)}
    rep = {"model_name": name, "model_type": "LP" if is_lp(model) else "MILP", "components": {}}
    groups = (("sets", pyo.Set), ("parameters", pyo.Param), ("variables", pyo.Var), ("constraints", pyo.Constraint),
              ("expressions", pyo.Expression), ("objectives", pyo.Objective))
    for g, kind in groups:
        rep["components"][g] = {}
        for c in model.component_objects(kind):
            d = {"description": DESC.get(c.name, "")}
            if c.is_indexed(): d["indexed_by"] = [s.name for s in c.index_set().subsets()]
            if g == "parameters": d["appears_on_lhs"] = c.name in lhs
            rep["components"][g][c.name] = d
    return rep

def all_component_names(rep): return [n for g in rep["components"].values() for n in g]

In [ ]:
# ---- Optional: paste the paper's EXACT Appendix C prompts here (from your own copy) to override the wording below. Leave empty otherwise. ----
EXACT = {}
# EXACT["coordinator"] = """...text with {agents} placeholder..."""

def fill(p, **kw):
    for k, v in kw.items(): p = p.replace("<<%s>>" % k, str(v)).replace("{%s}" % k, str(v))
    return p.replace("{{", "{").replace("}}", "}")

In [ ]:
# ---- Prompts: my wording of each Appendix C role (placeholders in <<double angle brackets>>) ----
PROMPTS = {}
PROMPTS["component_interpretation"] = """You are an operations-research expert. Explain a Pyomo model in PLAIN ENGLISH.
Pyomo code:
-----
<<code>>
-----
Components to describe:
-----
<<component_names>>
-----
For EVERY listed component give its physical meaning / intended use. Reply with ONE JSON object only, keys = component names, values = one-sentence descriptions. Do not omit any component."""

PROMPTS["model_illustration"] = """You are an operations-research expert introducing an optimization model to non-experts who know the application field but not optimization.
JSON representation of the model:
-----
<<json_representation>>
-----
Write a coherent, easy-to-follow introduction: what the problem is, who uses it and what it tries to achieve; the decisions (variables); the known data (parameters);
the constraints on the decisions; and the objective. Avoid jargon and symbols."""

PROMPTS["iis_inference"] = """You are an operations-research expert. The model below is INFEASIBLE. Your team computed its Irreducible Infeasible Subset (IIS):
-----
<<iis_info>>
-----
JSON representation of the model, to interpret the constraints and parameters:
-----
<<json_representation>>
-----
1) Say which constraints conflict and which parameters are involved. 2) Explain WHY they conflict, in physical terms, without jargon or symbols (formal tone).
3) Recommend parameters that can realistically be adjusted to restore feasibility (never physically immutable ones). 4) State the practical consequences of each recommendation."""

PROMPTS["coordinator"] = """You coordinate a team of optimization experts helping non-experts analyse an optimization model.
Team:
<<agents>>
Given the conversation, choose the next agent. Reply with ONLY this JSON (keys fixed): {"agent_name": "<agent>", "task": "<what it must do>"}.
If the user's question has already been fully answered, reply {"agent_name": "Explainer", "task": "DONE"}."""

PROMPTS["explainer"] = """You answer user questions about an optimization model in MARKDOWN. Users are domain experts, not optimization experts.
Give extra background only if they likely need it; otherwise be succinct. If engineers provided tool results, summarise them faithfully
(the user cannot see them) and NEVER state a number that does not appear in the tool results or the user's question."""

PROMPTS["syntax_reminder"] = """You are an operator working on a Pyomo model. Identify (a) the model component the user cares about and (b) the best function.
Choose ONLY from these functions:
1. feasibility_restoration: model is infeasible; user asks how much to change some parameter to restore feasibility.
2. components_retrieval: user wants current values of a set/parameter/variable/constraint/objective.
3. sensitivity_analysis: model feasible; user asks how the optimum reacts to a parameter WITHOUT stating how big the change is.
4. evaluate_modification: model feasible; user states a CONCRETE change (+10%, -20 units, two more...).
5. external_tools: user doubts the optimal solution / asks 'why not ...' and a new constraint must be added to test it.
Component names must be SYMBOLIC names; translate the user's words with this dictionary:
<<pairs>>
Reply with ONE JSON object: {"function": "<name>", "component": "<symbolic name or null>"}."""

PROMPTS["operator"] = """You are an optimization expert who invokes internal tools on a model. Reply with ONE JSON object {"function": "<name>", "args": {...}}.
Signatures:
 feasibility_restoration(component, index, direction)  # direction: up | down | both
 components_retrieval(component, index)
 sensitivity_analysis(component, index)
 evaluate_modification(component, index, operation, delta)  # operation: + | - | * | set (a 'tenth more' is operation "*" delta 1.1)
`index` = JSON list of set members in the component's declared order ([] means all). System reminder:
<<guidance>>"""

PROMPTS["programmer"] = """You write short Pyomo snippets for an optimization team. `model` is the already-built model: do NOT rebuild it, do NOT call a solver, do NOT import.
Available names: model, pyo, value, ConstraintList, sum, range. Use print(...) for information the user wants.
For a why-not / counterexample, ONLY ADD CONSTRAINTS that force the user's alternative (e.g. model.cf = ConstraintList(); model.cf.add(...)); the system re-solves afterwards.
Model context:
<<context>>

Worked example 1 — question: "Why is it not recommended to use just one supplier for roastery 2?"
```python
# the user believes a single supplier could serve roastery2; force at most one active supplier link for it
model.force_one_supplier = ConstraintList()
model.force_one_supplier.add(sum(model.z[s, 'roastery2'] for s in model.S) <= 1)
```
Worked example 2 — question: "Why is it not recommended to have production cost larger than transportation cost in the optimal setting?"
```python
# the user wants production cost to exceed transportation cost; force exactly that
model.counter_example = ConstraintList()
model.counter_example.add(model.production >= model.transportation)
```
Reply with ONE ```python block only, with comments explaining its purpose."""

PROMPTS["evaluator"] = """You review Pyomo code and its execution result written by a teammate for a user question.
Is the code bug-free and does it answer the user's query? Note: infeasibility caused by new counterfactual constraints is ACCEPTABLE
(the counterfactual may truly be infeasible). Reply with ONLY one JSON object: {"decision": "accept"|"reject", "comment": "..."}."""

PROMPTS.update(EXACT)

## Test fixtures (ground truth, hand-written)
Ground-truth objective values for the counterfactual (why-not) cases are computed by *our* tool layer from hand-written constraints. The first two cases are the two
worked examples shown inside the programmer prompt (**in-prompt = few-shot leakage, so easy**); the other four are held-out.

In [ ]:
def model_context(model_key):
    rep = json_representation(BUILD[model_key](), model_key)
    return json.dumps(rep)

WHYNOT = [  # (id, model, in_prompt?, query, ground-truth snippet)
 ("ex1", "coffee", True,  "Why is it not recommended to use just one supplier for roastery 2?",
  "model.cf = ConstraintList()\nmodel.cf.add(sum(model.z[s, 'roastery2'] for s in model.S) <= 1)"),
 ("ex2", "plant",  True,  "Why is it not recommended to have production cost larger than transportation cost in the optimal setting?",
  "model.cf = ConstraintList()\nmodel.cf.add(model.production >= model.transportation)"),
 ("h1",  "coffee", False, "Why not serve both roasteries only through supplier S3?",
  "model.cf = ConstraintList()\nfor r in model.R:\n    model.cf.add(model.z['S3', r] == 1)\nfor s in model.S:\n    if s != 'S3':\n        for r in model.R:\n            model.cf.add(model.z[s, r] == 0)"),
 ("h2",  "coffee", False, "Why don't we stop using supplier S1 altogether?",
  "model.cf = ConstraintList()\nfor r in model.R:\n    model.cf.add(model.z['S1', r] == 0)"),
 ("h3",  "plant",  False, "Why not ship nothing from plantB to the west region?",
  "model.cf = ConstraintList()\nmodel.cf.add(model.ship['plantB', 'west'] == 0)"),
 ("h4",  "plant",  False, "Why not make at least 500 units at plantB?",
  "model.cf = ConstraintList()\nmodel.cf.add(sum(model.ship['plantB', d] for d in model.D) >= 500)"),
]
GT = {}
for cid, mk, inp, q, snip in WHYNOT:
    r = run_counterfactual(BUILD[mk](), snip)
    GT[cid] = r
    print(f"{cid:4s} {mk:7s} base={r['objective_before']:.1f} counterfactual={r['objective_after']} ({r['status']})")

## 1. Component interpretation  (Illustrator, step 1)
The model must describe **every** component. Metric: fraction of components with a non-empty description (JSON must parse).

In [ ]:
rep = json_representation(BUILD["coffee"](), "coffee"); names = all_component_names(rep)
outs = run_all("component_interpretation", PROMPTS["component_interpretation"].replace("<<code>>", MODEL_SRC).replace("<<component_names>>", json.dumps(names)), "Proceed.", max_new_tokens=900)
show("§1 component interpretation (coffee model)", outs, 700)
for k in LLMS_KEYS:
    try:
        d = extract_json(outs[k]); cov = sum(1 for n in names if isinstance(d.get(n), str) and d[n].strip()) / len(names)
        record("1 component interpretation", k, "coffee", cov == 1.0, f"coverage={cov:.0%}")
    except Exception as e:
        record("1 component interpretation", k, "coffee", False, "invalid JSON")

## 2. Model illustration  (Illustrator, step 2)
Plain-English introduction for non-experts. Metric (crude): mentions the objective plus at least 3 of the domain entities. Read the outputs yourself.

In [ ]:
outs = run_all("model_illustration", PROMPTS["model_illustration"].replace("<<json_representation>>", json.dumps(rep)), "Proceed.", max_new_tokens=700)
show("§2 model illustration (coffee)", outs, 1200)
for k in LLMS_KEYS:
    t = outs[k].lower(); ent = sum(w in t for w in ["supplier", "roaster", "contract", "capacity", "kg"])
    record("2 model illustration", k, "coffee", ("cost" in t or "minimi" in t) and ent >= 3, f"entities={ent}")

## 3. IIS inference  (infeasible model)
The IIS (minimal conflicting constraint set) is computed by our deletion filter. Metric: explanation names a **capacity** side and a **need/demand** side of the conflict.

In [ ]:
mi = BUILD["coffee_infeasible"](); iis = compute_iis(mi); print("IIS:", iis)
rep_i = json_representation(mi, "coffee_infeasible")
outs = run_all("iis_inference", PROMPTS["iis_inference"].replace("<<iis_info>>", json.dumps(iis)).replace("<<json_representation>>", json.dumps(rep_i)), "Proceed.", max_new_tokens=700)
show("§3 IIS inference (coffee_infeasible)", outs, 1300)
for k in LLMS_KEYS:
    t = outs[k].lower()
    record("3 IIS inference", k, "coffee_infeasible", "capacity" in t and any(w in t for w in ["need", "demand", "require"]), "")

## 4. Coordinator  (routing JSON)
Solution-agnostic questions → Explainer; solution-specific → Engineer; finished conversation → `{"agent_name":"Explainer","task":"DONE"}`.

In [ ]:
AGENTS = "- Explainer: answers questions about the model description / general concepts directly, and is ALWAYS the final agent.\n- Engineer: runs solver-backed tools (values, sensitivity, what-if changes, infeasibility fixes, why-not counterfactuals)."
COORD = [  # (conversation, expected agent, expect DONE?)
 ("User: What is this model about?", "Explainer", False),
 ("User: What does the 'need' parameter mean?", "Explainer", False),
 ("User: By how much does total cost change if demand in the west region rises by 50 units?", "Engineer", False),
 ("User: How many kg does S1 ship to roastery1 in the plan?", "Engineer", False),
 ("User: Why not use only supplier S3?", "Engineer", False),
 ("User: What if demand in the east region rises by 100?\nEngineer: tool result: total cost rises from 12,400 to 13,300.\nExplainer: Raising east demand by 100 units increases total cost by 900 (from 12,400 to 13,300).", "Explainer", True),
]
rows = []
for conv, exp_agent, exp_done in COORD:
    outs = run_all("coordinator", fill(PROMPTS["coordinator"], agents=AGENTS), "Model: " + "optimization model (suppliers/roasteries or plants/regions).\nConversation:\n" + conv, max_new_tokens=120)
    for k in LLMS_KEYS:
        try:
            r = extract_json(outs[k]); ok = r.get("agent_name") == exp_agent and ((str(r.get("task", "")).strip().upper() == "DONE") == exp_done)
        except Exception: r, ok = outs[k][:60], False
        record("4 coordinator", k, conv.splitlines()[0][:50], ok, r if not ok else "")
        rows.append((k, conv.splitlines()[0][:55], exp_agent + (" (DONE)" if exp_done else ""), str(r)[:70], ok))
print(pd.DataFrame(rows, columns=["model", "case", "expected", "got", "ok"]).to_string(index=False))

## 5. Explainer  (numbers must come from the tool)
We execute a real what-if on the plant model and ask the explainer to report it. Metric: every number in the answer (ignoring small integers) must appear in the
tool output or the question. Derived numbers such as percentages are flagged on purpose — inspect them.

In [ ]:
def run_tool(model_key, call):
    f = TOOLFN[call["function"]]; a = dict(call["args"]); comp = a.pop("component", None)
    return f(BUILD[model_key](), comp, **a)

TOOLFN = {"feasibility_restoration": feasibility_restoration, "components_retrieval": components_retrieval,
          "sensitivity_analysis": sensitivity_analysis, "evaluate_modification": evaluate_modification}

def nums(text):
    out = []
    for s in re.findall(r"\d[\d,]*\.?\d*", text):
        s = s.replace(",", "").rstrip(".")
        try: out.append((float(s), "." in s))
        except ValueError: pass
    return out

def ungrounded(answer, allowed_text):
    allowed = [v for v, _ in nums(allowed_text)]
    allowed += [round(v, d) for v in allowed for d in (0, 1, 2)]
    bad = []
    for v, is_dec in nums(answer):
        if not is_dec and v <= 12: continue
        tol = 0.051 if is_dec else 0.51
        if not any(abs(v - a) <= tol for a in allowed): bad.append(v)
    return bad

q5 = "What if the demand in the east region rises by 100 units?"
tool5 = run_tool("plant", {"function": "evaluate_modification", "args": {"component": "demand", "index": ["east"], "operation": "+", "delta": 100}})
print("tool output:", tool5)
outs = run_all("explainer", PROMPTS["explainer"], f"User question: {q5}\nTool results: {json.dumps(tool5)}", max_new_tokens=350)
show("§5 explainer", outs, 900)
for k in LLMS_KEYS:
    bad = ungrounded(outs[k], json.dumps(tool5) + " " + q5)
    record("5 explainer (grounded numbers)", k, "plant what-if", not bad, f"ungrounded={bad}")

## 6. Syntax reminder  (pick the function + component)
Five query types, one each. Metric: correct function **and** correct symbolic component (not needed for `external_tools`).

In [ ]:
SYN = [  # (model, query, function, component)
 ("coffee_infeasible", "How much should the beans needed by roastery2 change to make this feasible?", "feasibility_restoration", "need"),
 ("coffee", "How many kg can supplier S3 ship in total?", "components_retrieval", "capacity_in_supplier"),
 ("plant", "How sensitive is total cost to demand in the west region?", "sensitivity_analysis", "demand"),
 ("plant", "What if the demand in the east region rises by 100 units?", "evaluate_modification", "demand"),
 ("coffee", "Why don't we serve roastery1 only through supplier S2?", "external_tools", None),
]
def pairs_for(mk):
    r = json_representation(BUILD[mk](), mk); return json.dumps({n: d["description"] for g in r["components"].values() for n, d in g.items() if d["description"]})
rows = []
for mk, q, f_exp, c_exp in SYN:
    outs = run_all("syntax_reminder", PROMPTS["syntax_reminder"].replace("<<pairs>>", pairs_for(mk)), f"Model status: {'infeasible' if 'infeasible' in mk else 'feasible'}.\nUser query: {q}", max_new_tokens=80)
    for k in LLMS_KEYS:
        try:
            r = extract_json(outs[k]); ok = r.get("function") == f_exp and (c_exp is None or r.get("component") == c_exp)
        except Exception: r, ok = outs[k][:50], False
        record("6 syntax reminder", k, f_exp, ok, "" if ok else r)
        rows.append((k, f_exp, str(r)[:80], ok))
print(pd.DataFrame(rows, columns=["model", "expected function", "got", "ok"]).to_string(index=False))

## 7. Operator  (emit arguments, then really run the tool)
A deterministic "syntax guidance" string (valid index structure, LHS warning) is injected, as the paper's reminder does. Metric: arguments match ground truth **and** the tool runs without error.

In [ ]:
def syntax_guidance(mk, component):
    model = BUILD[mk](); c = getattr(model, component, None)
    if c is None: return f"Unknown component {component}."
    rep = json_representation(model, mk)
    d = next(g[component] for g in rep["components"].values() if component in g)
    ex = [list(k) if isinstance(k, tuple) else [k] for k in list(c.keys())[:3]] if c.is_indexed() else None
    return (f"`{component}` ({d.get('description','')}); indexed by {d.get('indexed_by', 'nothing')}; valid index examples: {ex}. "
            "Pass index as a JSON list of members in that order, or [] for all." + (" WARNING: appears on LHS of constraints." if d.get("appears_on_lhs") else ""))

OPS = [  # (model, query, function, gt args)
 ("coffee_infeasible", "How much should the beans needed by roastery2 change to make this feasible?", "feasibility_restoration", {"component": "need", "index": ["roastery2"]}),
 ("coffee", "How many kg can supplier S3 ship in total?", "components_retrieval", {"component": "capacity_in_supplier", "index": ["S3"]}),
 ("plant", "How sensitive is total cost to demand in the west region?", "sensitivity_analysis", {"component": "demand", "index": ["west"]}),
 ("plant", "What if the demand in the east region rises by 100 units?", "evaluate_modification", {"component": "demand", "index": ["east"], "operation": "+", "delta": 100}),
]
def args_match(a, g):
    if a.get("component") != g["component"]: return False
    if [str(x) for x in (a.get("index") or [])] != [str(x) for x in g["index"]]: return False
    if "operation" in g: return a.get("operation") == g["operation"] and abs(float(a.get("delta", "nan")) - g["delta"]) < 1e-9
    return True
rows = []
for mk, q, f_exp, g in OPS:
    outs = run_all("operator", PROMPTS["operator"].replace("<<guidance>>", syntax_guidance(mk, g["component"])), f"Function chosen by the reminder: {f_exp}\nUser query: {q}", max_new_tokens=150)
    for k in LLMS_KEYS:
        try:
            call = extract_json(outs[k]); a = call.get("args", {})
            res = run_tool(mk, {"function": call["function"], "args": a})
            ok = call["function"] == f_exp and args_match(a, g) and "error" not in res
            note = "" if ok else f"{call}"
        except Exception as e:
            call, res, ok, note = outs[k][:80], None, False, f"exception {type(e).__name__}"
        record("7 operator + tool", k, f_exp, ok, note)
        rows.append((k, f_exp, json.dumps(call.get("args") if isinstance(call, dict) else call)[:80], ok))
print(pd.DataFrame(rows, columns=["model", "function", "args produced", "ok"]).to_string(index=False))

## 8. Code reminder  (program scaffold)
The programmer's code is spliced into the model source at the marker line; the **solve block is ours** and the LLM is told not to change it. This cell proves the scaffold runs.

In [ ]:
SOLVE_BLOCK = """
# ---- standard solve block (system-owned; the LLM must not touch it) ----
_st, _obj = solve(model)
print("Status:", _st, "| Objective:", _obj)
"""
def assemble_program(model_fn, snippet):
    return MODEL_SRC + f"\nmodel = {model_fn}()\n# YOUR CODE GOES HERE\n" + snippet + "\n" + SOLVE_BLOCK
prog = assemble_program("build_coffee", WHYNOT[0][4])
print(prog[-700:])
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    exec(prog, {**globals(), "ConstraintList": pyo.ConstraintList})
print("scaffold output ->", buf.getvalue().strip())
record("8 code reminder (scaffold)", "-", "assemble+exec", "Objective" in buf.getvalue(), "")

## 9. Programmer  (counterfactual constraints; includes the two worked examples)
The model writes constraints only; our solve block re-solves. Metric: counterfactual **objective equals ground truth** (both infeasible also counts). `ex1`/`ex2` are in the prompt (few-shot leakage).

In [ ]:
def snippet_ok(mk, snippet, gt):
    r = run_counterfactual(BUILD[mk](), snippet)
    a, b = r["objective_after"], gt["objective_after"]
    return ((a is None and b is None) or (a is not None and b is not None and math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-6))), r

PROG_OUT = {}
rows = []
for cid, mk, inp, q, _ in WHYNOT:
    sysmsg = PROMPTS["programmer"].replace("<<context>>", model_context(mk))
    outs = run_all("programmer", sysmsg, q, max_new_tokens=400)
    for k in LLMS_KEYS:
        code_k = extract_code(outs[k]); PROG_OUT[(cid, k)] = code_k
        try: ok, r = snippet_ok(mk, code_k, GT[cid]); note = f"got={r['objective_after']} gt={GT[cid]['objective_after']}"
        except Exception as e: ok, note = False, f"{type(e).__name__}: {str(e)[:80]}"
        record("9 programmer" + (" (in-prompt)" if inp else " (held-out)"), k, cid, ok, note)
        rows.append((k, cid, "in-prompt" if inp else "held-out", ok, note))
print(pd.DataFrame(rows, columns=["model", "case", "kind", "ok", "note"]).to_string(index=False))
print("\nExample generated code (llama/qwen, case h1):")
for k in LLMS_KEYS: print(f"--- {k} ---\n{PROG_OUT[('h1', k)]}")

## 10. Evaluator  (accept / reject)
Three scenarios from real executions: correct code (accept), buggy code that raises an error (reject), and correct code whose counterfactual is genuinely infeasible (**accept** — the paper's special rule).

In [ ]:
good_snip = WHYNOT[3][4]
bug_snip = "model.cf = ConstraintList()\nmodel.cf.add(model.nonexistent_variable['S1'] <= 1)"
inf_snip = "model.cf = ConstraintList()\nmodel.cf.add(sum(model.z[s, 'roastery2'] for s in model.S) <= 0)"
def exec_result(mk, snippet):
    try: return json.dumps(run_counterfactual(BUILD[mk](), snippet))
    except Exception as e: return json.dumps({"error": f"{type(e).__name__}: {e}"})
EV = [("good code", good_snip, "Why don't we stop using supplier S1 altogether?", "accept"),
      ("buggy code", bug_snip, "Why don't we stop using supplier S1 altogether?", "reject"),
      ("acceptably infeasible", inf_snip, "Why not leave roastery2 without any supplier link?", "accept")]
rows = []
for name, snip, q, exp in EV:
    res = exec_result("coffee", snip)
    outs = run_all("evaluator", PROMPTS["evaluator"], f"User query: {q}\nCode:\n```python\n{snip}\n```\nExecution result: {res}", max_new_tokens=150)
    for k in LLMS_KEYS:
        try: r = extract_json(outs[k]); ok = r.get("decision") == exp
        except Exception: r, ok = outs[k][:60], False
        record("10 evaluator", k, name, ok, "" if ok else r)
        rows.append((k, name, exp, str(r)[:70], ok))
print(pd.DataFrame(rows, columns=["model", "scenario", "expected", "got", "ok"]).to_string(index=False))

## 11. Full loop: coordinator → engineer → explainer
Everything wired together (reminder → operator → tool, or programmer ↔ evaluator for why-not). Skipped when `FAST = True`.
Metric: right function, tool/solver succeeded, and for why-not the counterfactual objective equals ground truth.

In [ ]:
def team_answer(key, mk, query, max_loops=3):
    trace = {"query": query}
    summary = "Optimization model: " + json.dumps({n: d["description"] for g in json_representation(BUILD[mk](), mk)["components"].values() for n, d in g.items() if d["description"]})[:900]
    route = extract_json(generate(key, "coordinator", fill(PROMPTS["coordinator"], agents=AGENTS), f"{summary}\nConversation:\nUser: {query}", None, 120))
    trace["route"] = route.get("agent_name"); result = None
    if trace["route"] == "Engineer":
        pick = extract_json(generate(key, "syntax_reminder", PROMPTS["syntax_reminder"].replace("<<pairs>>", pairs_for(mk)),
                                     f"Model status: {'infeasible' if 'infeasible' in mk else 'feasible'}.\nUser query: {query}", None, 80))
        trace["function"] = pick.get("function")
        if pick.get("function") != "external_tools":
            call = extract_json(generate(key, "operator", PROMPTS["operator"].replace("<<guidance>>", syntax_guidance(mk, pick.get("component"))),
                                         f"Function chosen by the reminder: {pick['function']}\nUser query: {query}", None, 150))
            trace["args"] = call.get("args"); result = run_tool(mk, {"function": call["function"], "args": call.get("args", {})})
        else:
            fb = ""
            for _ in range(max_loops):
                snip = extract_code(generate(key, "programmer", PROMPTS["programmer"].replace("<<context>>", model_context(mk)), query + (f"\nEvaluator feedback: {fb}" if fb else ""), None, 400))
                try: result = run_counterfactual(BUILD[mk](), snip)
                except Exception as e: result = {"error": f"{type(e).__name__}: {e}"}
                v = extract_json(generate(key, "evaluator", PROMPTS["evaluator"], f"User query: {query}\nCode:\n```python\n{snip}\n```\nExecution result: {json.dumps(result)}", None, 150))
                trace["code"] = snip
                if v.get("decision") == "accept": break
                fb = v.get("comment", "")
    trace["result"] = result
    trace["answer"] = generate(key, "explainer", PROMPTS["explainer"], f"User question: {query}\nTool results: {json.dumps(result, default=str)}", None, 350)
    return trace

E2E = [  # (model, query, expected function, gt-key for why-not or None)
 ("coffee_infeasible", "The model has no solution. How much should the beans needed by roastery2 change to fix that?", "feasibility_restoration", None),
 ("plant", "What if the demand in the east region rises by 100 units?", "evaluate_modification", None),
 ("plant", "How sensitive is total cost to demand in the west region?", "sensitivity_analysis", None),
 ("plant", "Why not make at least 500 units at plantB?", "external_tools", "h4"),
]
if FAST:
    print("FAST=True: skipping §11")
else:
    for mk, q, f_exp, gtk in E2E:
        for k in LLMS_KEYS:
            try:
                tr = team_answer(k, mk, q)
                ok = tr.get("route") == "Engineer" and tr.get("function") == f_exp and "error" not in (tr["result"] or {})
                if ok and gtk:
                    a, b = tr["result"]["objective_after"], GT[gtk]["objective_after"]
                    ok = (a is None and b is None) or (a is not None and b is not None and math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-6))
                print("=" * 100); print(f"[{k}] {q}\n  route={tr.get('route')} function={tr.get('function')} ok={ok}\n  ANSWER: {tr['answer'][:600]}")
            except Exception as e:
                ok = False; print(f"[{k}] {q} -> FAILED {type(e).__name__}: {str(e)[:120]}")
            record("11 full loop", k, f_exp, ok, "")

## Scoreboard
Pass rates per Appendix C item and model. **n is tiny** (1–6 cases per item): use it to see *where* each model breaks (JSON discipline, index errors, direction of a constraint,
hallucinated numbers), not to rank them. The `note` column of the table below shows the first failures.

In [ ]:
df = pd.DataFrame(RESULTS)
board = df.pivot_table(index="section", columns="model", values="passed", aggfunc=lambda s: f"{int(s.sum())}/{len(s)}")
print(board.to_string())
print("\nOverall:", {k: f"{int(g.passed.sum())}/{len(g)}" for k, g in df[df.model != "-"].groupby("model")})
fails = df[~df.passed]
print("\nFailures (first 25):"); print(fails[["section", "model", "case", "note"]].head(25).to_string(index=False))
df.to_csv("scoreboard.csv", index=False); print("\nsaved scoreboard.csv")

## Where to go next
* Swap in the paper's exact Appendix C wording via the `EXACT` override cell and compare.
* Add your own models/queries to `WHYNOT`, `SYN`, `OPS` and `COORD`; keep ground truth hand-written.
* Re-run with `temperature` sampling and several seeds if you want variance estimates; this notebook is greedy (deterministic).
* Known limitations: HiGHS reports a generic "infeasible or no solution" status; the IIS is a deletion filter over constraints (not variable bounds);
  Pyomo's dual sign convention was verified against finite differences only on LPs.